# MUR SST

Multi-scale Ultra-high Resolution [(MUR) Sea Surface Temperature](https://podaac.jpl.nasa.gov/MEaSUREs-MUR) is produced by scientists at NASA's Jet Propulsion Laboratory who are part of an international science team - The Group for High Resolution Sea Surface Temperature [GHRSST](https://www.ghrsst.org/). MUR SST are from 2002 - present as global, daily, 4 km spatial resolution maps. This makes them great for looking at coastal dynamics where there is a lot of small-scale variablity.  MUR SST takes measurements from many different satellites and blends them all together to create one, daily, gap-free, map of SST.

There are a couple ways to access the data, but since the full dataset is about 4,000 GB, it is easier to just access what you need on the cloud rather than spend a few weeks downloading it.

If you don't need recent data, the easiest way to get [MUR SST is on AWS](https://registry.opendata.aws/mur/) where it is stored in a special cloud optimized format called Zarr. There are some tutorials that show you how to quickly access the data.

If you need more recent data, this notebook shows you how to access all the files, but it is slower because of the format the data is stored in.

In this code I'm testing out appending to a zarr store

I tested whether it was still necessary to set decode_cf=False in the xarray read. Zarr uses the encodings, you can see them in the .zarr file in each variable directory where it sets how to save the data. This is not necessary anymore.


In [1]:
#from urllib import request
#from http.cookiejar import CookieJar
import s3fs
import requests
from os.path import dirname, join
from io import StringIO
import xarray as xr
#import earthaccess
import datetime
import os
import matplotlib.pyplot as plt
import tempfile
import time
import pandas as pd
import glob
import re

# Function to sort files numerically
def numerical_sort(value):
    parts = re.split(r'(\d+)', value)
    return [int(text) if text.isdigit() else text.lower() for text in parts]

#SET YOUR LOCATION HERE
#center_lat, center_lon = 37.820220, -122.480255

# because accessing the data via the internet can be a little bit hiccup-y we
# have a special way to write the data that is more robust than just trying to 
# write it out. this is because the data isn't actually loaded to your computer
# until it is needed, for example when you want to save it locally.
fpath = 'c:/data_sf/'
fpath_zarr = 'c:/data_sf_zarr/'
files = sorted(glob.glob(fpath+'*.nc'), key=numerical_sort)
fname_zarr = fpath_zarr + 'mur_sst_summer_sol_test.zarr'
fname_zarr_all = fpath_zarr + 'mur_sst_summer_sol_all.zarr'

In [2]:
ds=xr.open_mfdataset(files[0:36], combine="by_coords")
#rechunck data      #cci SUGGESTION
itime_chunk = 5   #200
ilat_chunk = 157    #300
ilon_chunk = 180    #600
ds_chunked = ds.chunk({'time':itime_chunk,'lat':ilat_chunk,'lon':ilon_chunk})
#output data
ds_chunked.to_zarr(fname_zarr,consolidated=True)

ContainsGroupError: path '' contains a group

In [ ]:
#read in netcdf
ds=xr.open_mfdataset(files[0:36])
ds = ds.load()
#read in zarr store
ds_zarr = xr.open_zarr(fname_zarr)
ds_zarr = ds_zarr.load()
#ds_zarr

In [ ]:
for i in range(0,90,10):
    print(ds.analysed_sst[i,0,0].data,ds_zarr.analysed_sst[i,0,0].data)
    print(ds.analysed_sst[i,-1,-1].data,ds_zarr.analysed_sst[i,-1,-1].data)

In [ ]:
# okay this checks out, can write and zarr file is the same

In [ ]:
# read in old zarr store and append new data onto it
#fname_zarr = fpath_zarr + 'mur_sst_summer_sol2.zarr'
#ds_zarr = xr.open_zarr(fname_zarr)
ds=xr.open_mfdataset(files[36:74], combine="by_coords")
#rechunck data      #cci SUGGESTION
itime_chunk = 5   #200
ilat_chunk = 157    #300
ilon_chunk = 180    #600
ds_chunked = ds.chunk({'time':itime_chunk,'lat':ilat_chunk,'lon':ilon_chunk})
#output data
#ds_chunked.to_zarr(fname_zarr,consolidated=True)
ds_chunked.to_zarr(fname_zarr, mode="a", append_dim="time")

In [ ]:
ds=xr.open_mfdataset(files, combine="by_coords")
#rechunck data      #cci SUGGESTION
itime_chunk = 5   #200
ilat_chunk = 157    #300
ilon_chunk = 180    #600
ds_chunked = ds.chunk({'time':itime_chunk,'lat':ilat_chunk,'lon':ilon_chunk})
#output data
#ds_chunked.to_zarr(fname_zarr,consolidated=True)
ds_chunked.to_zarr(fname_zarr_all)

In [ ]:
# check if appended data still okay

In [ ]:
ds=xr.open_mfdataset(files, combine="by_coords")
ds_zarr = xr.open_zarr(fname_zarr)
ds_zarr_all = xr.open_zarr(fname_zarr_all)
ds = ds.load()
ds_zarr = ds_zarr.load()
ds_zarr_all = ds_zarr_all.load()

In [ ]:
for i in range(0,366,20):
    print(ds.analysed_sst[i,0,0].data,ds_zarr.analysed_sst[i,0,0].data,ds_zarr_all.analysed_sst[i,0,0].data)
    print(ds.analysed_sst[i,-1,-1].data,ds_zarr.analysed_sst[i,-1,-1].data,ds_zarr_all.analysed_sst[i,-1,-1].data)